# Modeling Baselines


In this baseline modeling process, we perform the following sequence of steps:

1. We first preprocess the data. 
2. Impute missing values using the most frequent value for categorical variables and the median for numerical variables.
3. Numerical columns are then rescaled so they are distributed like N(0,1).
4. Perform a stratified split that preserves the grouping by patient_nbr since patients can have repeat visits to the hospital.
5. Fit a logistic regression model through the above pipeline and the use it to make predictions. 
6. Evaluate the performance of the model using various metrics. Recall from our EDA that our target metric is F1-Score.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, log_loss)

c:\Users\daoke\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\daoke\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('../data/diabetic_data.csv')

display(df.head())
print(df.columns)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')


In [3]:
# Data preprocessing based off of previous EDA
# Will put this Data Preprocessing into a function to be used in the final model

# Basic preprocessing function to clean data.

def preprocess_data(df):

    # Drop encounter_id, payer_code
    df = df.drop(['encounter_id', 'payer_code'], axis=1)

    # Lowercase all Column Names
    df.columns = df.columns.str.lower()

    # Edit the readmitted column to be a binary target. We are focused on patients readmitted within 30 days.
    df["readmitted"] = df["readmitted"].replace({">30": 0, "<30": 1, "NO": 0})

    df.rename(columns={"readmitted":"target"},inplace=True)
    df["target"].astype(int)
    
    return df

# Extract numerical and categorical columns

def num_cat(df,target):
    # Identify numerical and categorical columns
    df = df.drop(columns=[target])
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    return num_cols, cat_cols

In [4]:
df = preprocess_data(df)

In [5]:
num_cols, cat_cols = num_cat(df,"target")
features = num_cols+cat_cols

X = df[features]
y = df['target'].astype(int) 

C:\Users\daoke\AppData\Local\Temp\ipykernel_7800\148657544.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


In [6]:
# The target is imbalanced. We also want to keep repeat patients together.

# 1. Set n_splits=5 so the test size naturally becomes 1/5 = 20%
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Get the indices for the very first split next() gets only the first split for us
train_idx, test_idx = next(sgkf.split(X, y, groups=X['patient_nbr']))

# 3. Create a single train/test split
Xt, Xv = X.iloc[train_idx], X.iloc[test_idx]
yt, yv = y.iloc[train_idx], y.iloc[test_idx]

In [7]:
# We will make a Pipeline for the Baseline Model with a Preprocessing Step and a Logistic Regression Model.
# The preprocessing should only be done using the training data.

# A naive baseline will be to simply use a median and most frequent imputer for the numerical and categorical columns respectively.

num = Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))])
prep = ColumnTransformer([("num",num,num_cols),("cat",cat,cat_cols)])
clf = Pipeline([("prep", prep),("model", LogisticRegression(max_iter=1000))])


In [8]:
# Fit the model to the training data

clf.fit(Xt,yt)

# Have the model now make predictions

labels = clf.predict(Xv)                   
probs = clf.predict_proba(Xv)
p = probs[:, 1]

# Analyze the accuracy, roc_auc, log_loss, precision, recall, F1-score. We also can look at the confusion matrix.

acc = accuracy_score(yv, labels)
auc = roc_auc_score(yv, p)               
loss = log_loss(yv, probs, labels=[0,1])

custom = (p >= 0.10).astype(int)          # example threshold where we give a 1 if and only if p>=0.10 I played around with p until I got a high f1 score
precision = precision_score(yv, custom, zero_division=0)
recall = recall_score(yv, custom, zero_division=0)
f1 = f1_score(yv, custom, zero_division=0)

tn, fp, fn, tp = confusion_matrix(yv, custom, labels=[0, 1]).ravel()

print(f"The results of the baseline model is:") 
print(f"Accuracy: {np.round(acc,3)}")
print(f"AUC: {np.round(auc,3)}")
print(f"Log Loss: {np.round(loss,3)}\n\n")
print(f"Assuming a threshold of p=0.35:")
print(f"Precision: {np.round(precision,3)}")
print(f"Recall: {np.round(recall,3)}")
print(f"F1-Score: {np.round(f1,3)}")

The results of the baseline model is:
Accuracy: 0.887
AUC: 0.638
Log Loss: 0.338


Assuming a threshold of p=0.35:
Precision: 0.153
Recall: 0.622
F1-Score: 0.245


In [9]:
print(f"If we were to simply predict that the discharged patient would not return within")
print(f"30 days no matter what the model would get it correct {100*np.round(1 - sum(yv)/len(yv),3)}% of the time.")

If we were to simply predict that the discharged patient would not return within
30 days no matter what the model would get it correct 88.8% of the time.


# Conclusion

We see here that we have a very low F1-score of 0.245. Our precision is also a bit low. The accuracy is naturally very high and similar to the dummy guess which gets 0.888 due to the unbalanced nature of the target. The AUC score is 0.637 which indicates that the model performs better than a 50/50 guess, but this is not shocking due to the unbalanced nature.

What this shows is that the baseline model is pretty bad and there can be room for significant improvement.